In [1]:
# ==============================================================================
# CELL 0: OPUS CLOUD KERNEL (Cloud-Native Bootstrap)
# ==============================================================================
# Purpose: 1. Authenticate to GCP.
#          2. Define Sovereign Project IDs.
#          3. Initialize the BigQuery Client for all subsequent operations.
#          4. Apply the Opus Lab Visual Canon.
# ==============================================================================

# --- 1. HYGIENE & IMPORTS ---
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import auth
from google.cloud import bigquery # <--- El nuevo motor

# --- 2. CLOUD CONFIGURATION (The Master Keys) ---
# Ahora usamos la jerarquía de nombres correcta:
PROJECT_ID = 'drivers-dilemma'
DATASET_REAL = 'pienza_mini'    # Contiene el Star Schema (Realidad)
DATASET_SYNTH = 'pienza_big'  # Contiene el Manifold (Simulación)

# --- 3. AUTHENTICATION & CLIENT INITIALIZATION ---
print("🔐 Iniciando Solicitud de Soberanía Cloud (Autenticación GCP)...")
try:
    auth.authenticate_user()
    print("✅ Autenticación de Usuario Exitosa.")

    client = bigquery.Client(project=PROJECT_ID)
    print(f"🚀 Cliente BigQuery Activo en proyecto: {PROJECT_ID}")

except Exception as e:
    print(f"❌ FALLO CRÍTICO DE CONEXIÓN: {e}")
    raise

# --- 4. VISUAL CANON (OPUS LAB THEME) ---
OPUS_PURPLE = '#440154'
OPUS_TEAL   = '#21918c'
OPUS_GREY   = '#FAFAFA'
OPUS_TEXT   = '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': OPUS_GREY,
    'axes.facecolor': OPUS_GREY,
    'text.color': OPUS_TEXT,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.edgecolor': '#DDDDDD',
    'grid.color': '#E0E0E0',
    'font.family': 'sans-serif',
    'axes.titlecolor': OPUS_PURPLE,
    'axes.titleweight': 'bold',
    'figure.titlesize': 24,
    'figure.titleweight': 'bold'
})

print("✅ Visual Identity Loaded: Opus Lab (Cloud Mode).")
print(f"🔗 Ready to operate on {DATASET_REAL} (Reality) and {DATASET_SYNTH} (Simulation).")
print("\n--- SYSTEM READY ---")

🔐 Iniciando Solicitud de Soberanía Cloud (Autenticación GCP)...
✅ Autenticación de Usuario Exitosa.
🚀 Cliente BigQuery Activo en proyecto: drivers-dilemma
✅ Visual Identity Loaded: Opus Lab (Cloud Mode).
🔗 Ready to operate on pienza_mini (Reality) and pienza_big (Simulation).

--- SYSTEM READY ---


In [2]:
# ==============================================================================
# CELL 1 (FIXED): CREAR DATASET + TABLA EXTERNA (ZERO COST)
# ==============================================================================

from google.colab import drive
from googleapiclient.discovery import build
from google.auth import default
import os

# --- 1. CONFIGURACIÓN ---
FILE_PATH_DRIVE = '/content/drive/MyDrive/_Pienza/Assets/Phase_6/df_gan_training_set_v8.parquet'
FILENAME = 'df_gan_training_set_v8.parquet'

DEST_DATASET = DATASET_SYNTH  # 'pienza_big'
DEST_TABLE = 'gan_training_set_v8_reference'
TABLE_ID = f"{PROJECT_ID}.{DEST_DATASET}.{DEST_TABLE}"

# --- 2. ASEGURAR DATASET (Solución al error 404) ---
def ensure_dataset_exists():
    dataset_ref = client.dataset(DEST_DATASET)
    try:
        client.get_dataset(dataset_ref)
        print(f"✅ Dataset '{DEST_DATASET}' ya existe.")
    except Exception:
        print(f"🏗️ Dataset '{DEST_DATASET}' no encontrado. Creándolo...")
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        client.create_dataset(dataset)
        print(f"✅ Dataset '{DEST_DATASET}' creado exitosamente.")

# --- 3. CREAR TABLA EXTERNA ---
def create_external_table():
    # Obtener ID de Drive (Reutilizamos tu lógica que ya funcionó)
    creds, _ = default()
    service = build('drive', 'v3', credentials=creds)
    results = service.files().list(
        q=f"name = '{FILENAME}' and trashed = false",
        fields="files(id, name)"
    ).execute()
    file_id = results.get('files', [])[0]['id']

    source_uri = f"https://drive.google.com/open?id={file_id}"

    # Configuración de Tabla Externa
    external_config = bigquery.ExternalConfig("PARQUET")
    external_config.source_uris = [source_uri]
    external_config.autodetect = True

    # Creamos/Reemplazamos la tabla
    table = bigquery.Table(TABLE_ID)
    table.external_data_configuration = external_config

    client.delete_table(TABLE_ID, not_found_ok=True)
    table = client.create_table(table)

    print(f"🔗 ENLACE EXITOSO: '{DEST_TABLE}' creado en '{DEST_DATASET}'.")
    print(f"💰 Costo de almacenamiento: $0.00 (Datos viven en Drive).")

# --- EJECUCIÓN ---
ensure_dataset_exists()
create_external_table()

✅ Dataset 'pienza_big' ya existe.
🔗 ENLACE EXITOSO: 'gan_training_set_v8_reference' creado en 'pienza_big'.
💰 Costo de almacenamiento: $0.00 (Datos viven en Drive).


In [3]:
# ==============================================================================
# CELL 2: DESPLIEGUE DEL MANIFOLD V7 (SIMULACIÓN)
# ==============================================================================
# Purpose: Conecta el Parquet generado por el GAN (V7) a BigQuery.
#          Mismo enfoque de Tabla Externa (Zero Cost).
# ==============================================================================

# 1. CONFIGURACIÓN DEL MANIFOLD V7
MANIFOLD_FILENAME = 'pienza_manifold_v8.parquet' # El archivo que generó tu GAN
TABLE_NAME_MANIFOLD = 'synthetic_manifold_v8'
MANIFOLD_TABLE_ID = f"{PROJECT_ID}.{DATASET_SYNTH}.{TABLE_NAME_MANIFOLD}"

def deploy_synthetic_universe_v7():
    print(f"\n🌉 Iniciando conexión del Manifold V8 a {DATASET_SYNTH}...")

    try:
        # A. Buscar el ID de Drive automáticamente
        creds, _ = default()
        service = build('drive', 'v3', credentials=creds)
        results = service.files().list(
            q=f"name = '{MANIFOLD_FILENAME}' and trashed = false",
            fields="files(id, name)"
        ).execute()

        items = results.get('files', [])
        if not items:
            raise FileNotFoundError(f"❌ No se encontró '{MANIFOLD_FILENAME}' en Drive.")

        file_id = items[0]['id']
        uri = f'https://drive.google.com/open?id={file_id}'
        print(f"✅ Archivo V8 detectado (ID: {file_id})")

        # B. Configurar Tabla Externa
        ext_config = bigquery.ExternalConfig("PARQUET")
        ext_config.source_uris = [uri]
        ext_config.autodetect = True

        table = bigquery.Table(MANIFOLD_TABLE_ID)
        table.external_data_configuration = ext_config

        # C. Despliegue (Delete & Create para refrescar puntero)
        client.delete_table(MANIFOLD_TABLE_ID, not_found_ok=True)
        client.create_table(table)

        print(f"✅ MANIFOLD V8 DESPLEGADO.")
        print(f"🔗 Tabla: {MANIFOLD_TABLE_ID}")
        print(f"💰 Costo: $0.00")

    except Exception as e:
        print(f"❌ ERROR EN EL DESPLIEGUE DEL MANIFOLD: {e}")

# EJECUTAR
deploy_synthetic_universe_v7()


🌉 Iniciando conexión del Manifold V8 a pienza_big...
✅ Archivo V8 detectado (ID: 1cg3IrWTQsV-xfxclPfQzGTZ5Uosd9Je-)
✅ MANIFOLD V8 DESPLEGADO.
🔗 Tabla: drivers-dilemma.pienza_big.synthetic_manifold_v8
💰 Costo: $0.00


In [4]:
# TEST RÁPIDO DE SOBERANÍA DE DATOS
for table_name in ['gan_training_set_v8_reference', 'synthetic_manifold_v8']:
    query = f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_SYNTH}.{table_name}`"
    count = client.query(query).to_dataframe().iloc[0]['total']
    print(f"✅ Tabla {table_name}: {count} filas detectadas.")

✅ Tabla gan_training_set_v8_reference: 5123 filas detectadas.
✅ Tabla synthetic_manifold_v8: 1010001 filas detectadas.


In [5]:
# ==============================================================================
# CELL 3: DESPLIEGUE DEL MANIFOLD DOWNSCALED (RESOLUCIÓN MICRO)
# ==============================================================================
# Purpose: Conecta el Parquet final con la resolución micro (Downscaled) a BigQuery.
#          Mismo enfoque de Tabla Externa (Zero Cost).
# ==============================================================================

# 1. CONFIGURACIÓN DEL MANIFOLD DOWNSCALED
DOWNSCALED_FILENAME = 'synthetic_manifold_v8_downscaled.parquet'
TABLE_NAME_DOWNSCALED = 'synthetic_manifold_v8_downscaled'
DOWNSCALED_TABLE_ID = f"{PROJECT_ID}.{DATASET_SYNTH}.{TABLE_NAME_DOWNSCALED}"

def deploy_downscaled_universe():
    print(f"\n🗺️ Iniciando conexión del Manifold Downscaled a {DATASET_SYNTH}...")

    try:
        # A. Buscar el ID de Drive automáticamente
        creds, _ = default()
        service = build('drive', 'v3', credentials=creds)
        results = service.files().list(
            q=f"name = '{DOWNSCALED_FILENAME}' and trashed = false",
            fields="files(id, name)"
        ).execute()

        items = results.get('files', [])
        if not items:
            raise FileNotFoundError(f"❌ No se encontró '{DOWNSCALED_FILENAME}' en Drive.")

        file_id = items[0]['id']
        uri = f'https://drive.google.com/open?id={file_id}'
        print(f"✅ Archivo Downscaled detectado (ID: {file_id})")

        # B. Configurar Tabla Externa
        ext_config = bigquery.ExternalConfig("PARQUET")
        ext_config.source_uris = [uri]
        ext_config.autodetect = True

        table = bigquery.Table(DOWNSCALED_TABLE_ID)
        table.external_data_configuration = ext_config

        # C. Despliegue (Delete & Create para refrescar puntero)
        client.delete_table(DOWNSCALED_TABLE_ID, not_found_ok=True)
        client.create_table(table)

        print(f"✅ MANIFOLD DOWNSCALED DESPLEGADO.")
        print(f"🔗 Tabla: {DOWNSCALED_TABLE_ID}")
        print(f"💰 Costo: $0.00")

    except Exception as e:
        print(f"❌ ERROR EN EL DESPLIEGUE DEL MANIFOLD DOWNSCALED: {e}")

# EJECUTAR
deploy_downscaled_universe()

# ==============================================================================
# AUDITORÍA FINAL: SOBERANÍA DE DATOS
# ==============================================================================
print("\n📊 AUDITORÍA FINAL DE TABLAS EN BIGQUERY (PIENZA BIG):")
print("-" * 65)

tablas_a_auditar = [
    'gan_training_set_v8_reference',
    'synthetic_manifold_v8',
    'synthetic_manifold_v8_downscaled'
]

for table_name in tablas_a_auditar:
    try:
        query = f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_SYNTH}.{table_name}`"
        count = client.query(query).to_dataframe().iloc[0]['total']
        print(f" ✅ Tabla {table_name}: {count:,} filas detectadas.")
    except Exception as e:
        print(f" ❌ Error al consultar {table_name}: {e}")

print("-" * 65)
print("🏁 ETL CLOUD COMPLETADO. EL ECOSISTEMA ESTÁ EN LÍNEA.")


🗺️ Iniciando conexión del Manifold Downscaled a pienza_big...
✅ Archivo Downscaled detectado (ID: 1mAK7cjEm7wVvaIXetY82eZxkJCUEPddn)
✅ MANIFOLD DOWNSCALED DESPLEGADO.
🔗 Tabla: drivers-dilemma.pienza_big.synthetic_manifold_v8_downscaled
💰 Costo: $0.00

📊 AUDITORÍA FINAL DE TABLAS EN BIGQUERY (PIENZA BIG):
-----------------------------------------------------------------
 ✅ Tabla gan_training_set_v8_reference: 5,123 filas detectadas.
 ✅ Tabla synthetic_manifold_v8: 1,010,001 filas detectadas.
 ✅ Tabla synthetic_manifold_v8_downscaled: 1,010,001 filas detectadas.
-----------------------------------------------------------------
🏁 ETL CLOUD COMPLETADO. EL ECOSISTEMA ESTÁ EN LÍNEA.
